# Greg Tollefson Math466 Assignment2

### Problem 1: Consider the matrix: 
$$
A =
\begin{bmatrix}
3.0 & 6.0 & -1.0 \\
-6.0 & -6.0 & 1.0 \\
2.0 & 1.0 & -1.0
\end{bmatrix}
$$

In [37]:
# Needed Libs
using LinearAlgebra
using SparseArrays 


In [58]:
# Helper functions: 

mutable struct OpCount
    add::Int
    sub::Int
    mul::Int
    div::Int
    sqrt::Int
end

OpCount() = OpCount(0, 0, 0, 0, 0)

function show_counts(name, ops)
    println(name)
    println("  Additions       = ", ops.add)
    println("  Subtractions    = ", ops.sub)
    println("  Multiplications = ", ops.mul)
    println("  Divisions       = ", ops.div)
    println("  Square roots    = ", ops.sqrt)
end

function counted_dot(x, y, ops)
    n = length(x)

    s = x[1] * y[1]
    ops.mul += 1

    for i = 2:n
        t = x[i] * y[i]
        ops.mul += 1

        s = s + t
        ops.add += 1
    end

    return s
end


function counted_norm(x, ops)
    n = length(x)

    s = x[1] * x[1]
    ops.mul += 1

    for i = 2:n
        t = x[i] * x[i]
        ops.mul += 1

        s = s + t
        ops.add += 1
    end

    ops.sqrt += 1
    return sqrt(s)
end


function counted_scale(a, x, ops)
    y = Vector{Float64}(undef, length(x))

    for i in eachindex(x)
        y[i] = a * x[i]
        ops.mul += 1
    end

    return y
end


function counted_sub(x, y, ops)
    z = Vector{Float64}(undef, length(x))

    for i in eachindex(x)
        z[i] = x[i] - y[i]
        ops.sub += 1
    end

    return z
end

function test_qr_solver(A, x_true, qr_method)

    # Construct b so the true solution is known
    b = A * x_true

    # Compute QR factorization
    Q, R = qr_method(A)

    # Solve Rx = Q'b
    y = Q' * b
    x = R \ y

    println("x_true = ")
    display(x_true)

    println("Computed x = ")
    display(x)

    println("Solution error = ", norm(x - x_true))
    println("Residual error = ", norm(A*x - b))

    #return x
end


test_qr_solver (generic function with 1 method)

#### Problem 1a)
Use Gram–Schmidt orthgonalization to write A = QR where Q isorthogonal and R is upper triangular.

In [56]:

function gramSchmidt_qr(A; count_ops=false, debug=false)

    A = Matrix{Float64}(A)
    m, n = size(A)

    if n != 3
        error("This implementation expects exactly 3 columns.")
    end

    # Create operation counter if requested
    if count_ops
        ops = OpCount()
    end

    # Extract columns of A
    a1 = A[:,1]
    a2 = A[:,2]
    a3 = A[:,3]


    # ---------------------------------------------------------
    # First orthonormal vector
    # ---------------------------------------------------------

    u1 = a1

    # e1 = u1 / ||u1||
    e1 = u1 / norm(u1)

    if count_ops
        # norm(u1)
        ops.mul += m
        ops.add += m - 1
        ops.sqrt += 1

        # scalar division of each component
        ops.div += m
    end


    # ---------------------------------------------------------
    # Second orthonormal vector
    # ---------------------------------------------------------

    # u2 = a2 - (a2'e1)e1
    u2 = a2 - dot(a2,e1)*e1

    if count_ops
        # dot(a2,e1)
        ops.mul += m
        ops.add += m - 1

        # scalar times e1
        ops.mul += m

        # subtract projection from a2
        ops.sub += m
    end

    # e2 = u2 / ||u2||
    e2 = u2 / norm(u2)

    if count_ops
        # norm(u2)
        ops.mul += m
        ops.add += m - 1
        ops.sqrt += 1

        # scalar division
        ops.div += m
    end


    # ---------------------------------------------------------
    # Third orthonormal vector
    # ---------------------------------------------------------

    # u3 = a3 - (a3'e1)e1 - (a3'e2)e2
    u3 = a3 - dot(a3,e1)*e1 - dot(a3,e2)*e2

    if count_ops
        # Two dot products
        ops.mul += 2*m
        ops.add += 2*(m - 1)

        # Two scalar-vector multiplications
        ops.mul += 2*m

        # Two vector subtractions
        ops.sub += 2*m
    end

    # e3 = u3 / ||u3||
    e3 = u3 / norm(u3)

    if count_ops
        # norm(u3)
        ops.mul += m
        ops.add += m - 1
        ops.sqrt += 1

        # scalar division
        ops.div += m
    end

    Q = [e1 e2 e3]

    # Check orthogonality: Q'Q ≈ I
    if debug
        CheckOrth = Q' * Q
        CheckOrth[abs.(CheckOrth) .< 1e-12] .= 0.0
        println("Check Orthogonality")
        display(CheckOrth)
    end


    R = Q' * A
    R[abs.(R) .< 1e-12] .= 0.0

    if count_ops
        # Q' is 3 × m and A is m × 3.
        #
        # Result has 9 entries.
        # Each entry is a dot product of length m:
        #
        #   m multiplications
        #   m-1 additions

        ops.mul += 9*m
        ops.add += 9*(m - 1)
    end

    if debug
        println("Show the R Matrix")
        display(R)
    end


    # Check A = QR
    if debug
        CheckA = Q * R
        println("Check that A = QR")
        display(CheckA)
    end

    if count_ops
        return Q, R, ops
    else
        return Q, R
    end

end



gramSchmidt_qr (generic function with 1 method)

In [61]:
A = [
     3.0   6.0  -1.0
    -6.0  -6.0   1.0
     2.0   1.0  -1.0
]

x_true = [1.0, 2.0, 3.0]

x = test_qr_solver(A, x_true, gramSchmidt_qr)

x_true = 


3-element Vector{Float64}:
 1.0
 2.0
 3.0

Computed x = 


3-element Vector{Float64}:
 0.9999999999999989
 2.000000000000001
 2.999999999999999

Solution error = 1.6764000044290905e-15
Residual error = 3.580361673049448e-15


### Problem 1b
Calculate the QR factorization of A using three Givens rotations.


In [40]:
function RotM(n,p,q,c,s)
    M=sparse(Float64,I,n,n)
    M[p,p]=c; M[p,q]=s
    M[q,p]=-s; M[q,q]=c
    return M
end

RotM (generic function with 1 method)

In [41]:
function givens_qr(A; count_ops=false)

    A = Matrix{Float64}(A)
    m, n = size(A)

    if m < n
        error("wide matrices not supported!")
    end

    Q = Matrix{Float64}(I, m, m)

    # Create an operation counter if requested
    if count_ops
        ops = OpCount()
    end

    for p = 1:n
        for q = m:-1:p+1

            r1 = A[p,p]
            r2 = A[q,p]

            z = sqrt(r1*r1 + r2*r2)

            if count_ops
                ops.mul += 2
                ops.add += 1
                ops.sqrt += 1
            end

            if z > 0
                c = r1/z
                s = r2/z

                if count_ops
                    ops.div += 2
                end
            else
                c = 1.0
                s = 0.0
            end

            M = RotM(m,p,q,c,s)

            A = M*A
            Q = Q*M'

            # Count applying the rotation to A,
            # but NOT construction of Q
            if count_ops
                active_cols = n - p + 1

                ops.mul += 4 * active_cols
                ops.add += active_cols
                ops.sub += active_cols
            end
        end
    end

    if count_ops
        return Q, A, ops
    else
        return Q, A
    end
end

givens_qr (generic function with 1 method)

In [62]:
A = [
     3.0   6.0  -1.0
    -6.0  -6.0   1.0
     2.0   1.0  -1.0
]

x_true = [1.0, 2.0, 3.0]

x = test_qr_solver(A, x_true, givens_qr)

x_true = 


3-element Vector{Float64}:
 1.0
 2.0
 3.0

Computed x = 


3-element Vector{Float64}:
 1.0000000000000016
 1.9999999999999991
 3.0000000000000018

Solution error = 2.5219419200973697e-15
Residual error = 2.5121479338940403e-15


### Problem 1c: 
Calculate the QR factorization using two Householder reflections.

In [43]:
function householder_qr(A; count_ops=false)

    A = Matrix{Float64}(A)
    m, n = size(A)

    if m < n
        error("wide matrices not supported!")
    end

    Q = Matrix{Float64}(I, m, m)

    # Create an operation counter only if flag is set
    if count_ops
        ops = OpCount()
    end

    for p = 1:min(n, m-1)

        x = A[p:m, p]
        z = norm(x)

        if count_ops
            k = length(x)
            ops.mul += k          # x[1]^2 + ... + x[k]^2
            ops.add += k - 1      # sum of the k squared terms
            ops.sqrt += 1         # sqrt(...)
        end
            

        if z > 0
            α = -copysign(z, x[1])

            v = copy(x)
            v[1] = v[1] - α  #  one subtraction 
            
                if count_ops
                    ops.sub += 1
                end
            
            v = v / norm(v)  # calculate norm then scalar division 
            
            if count_ops
                k = length(v)
    
                ops.mul += k
                ops.add += k - 1
                ops.sqrt += 1
                ops.div += k  
            end

            H = Matrix{Float64}(I, m, m)
            H[p:m, p:m] -= 2.0 * (v * v')  # outer product
            
                if count_ops
                    k = length(v)
    
                    ops.mul += 2 * k^2 
                    ops.sub += k^2
                end

            A = H * A
            
            if count_ops
                
                ops.mul += m^2 * n
                ops.add += m * n * (m - 1)
            end
            
            Q = Q * H # Ignore counts for forming Q
        end
    end

    if count_ops
        return Q, A, ops
    else
        return Q, A
    end
    
end

householder_qr (generic function with 1 method)

In [63]:
A = [
     3.0   6.0  -1.0
    -6.0  -6.0   1.0
     2.0   1.0  -1.0
]

x_true = [1.0, 2.0, 3.0]

x = test_qr_solver(A, x_true, householder_qr)

x_true = 


3-element Vector{Float64}:
 1.0
 2.0
 3.0

Computed x = 


3-element Vector{Float64}:
 1.0
 1.9999999999999991
 2.999999999999998

Solution error = 2.3914935841127266e-15
Residual error = 6.5418425621716e-15


### Problem 1d:
For each method count the number of additions, subtractions, mul-tiplications, divisions and square roots needed. Don’t include the effort needed to construct Q from the rotations or reflectors.

In [52]:
Qgs, Rgs, ops_gs = gramSchmidt_qr(A; count_ops=true)
Qg, Rg, ops_g = givens_qr(A; count_ops=true)
Qh, Rh, ops_h = householder_qr(A; count_ops=true)

show_counts("Gram-Schmidt", ops_gs)
show_counts("Givens", ops_g)
show_counts("Householder", ops_h)

Gram-Schmidt
  Additions       = 30
  Subtractions    = 9
  Multiplications = 54
  Divisions       = 9
  Square roots    = 3
Givens
  Additions       = 11
  Subtractions    = 8
  Multiplications = 38
  Divisions       = 6
  Square roots    = 3
Householder
  Additions       = 42
  Subtractions    = 15
  Multiplications = 90
  Divisions       = 5
  Square roots    = 4


x_true = 


3-element Vector{Float64}:
 1.0
 2.0
 3.0

Computed x = 


3-element Vector{Float64}:
 0.9999999999999989
 2.000000000000001
 2.999999999999999

Solution error = 1.6764000044290905e-15
Residual error = 3.580361673049448e-15


### Problem 2: Consider the matrix

$$
A =
\begin{bmatrix}
-5 & 6 & 7 & -5 \\
-7 & 5 & -9 & -3 \\
-3 & 7 & -6 & -1
\end{bmatrix}
\in \mathbb{R}^{3 \times 4}.
$$


In [66]:
A = [
    -5   6   7  -5
    -7   5  -9  -3
    -3   7  -6  -1
]

3×4 Matrix{Int64}:
 -5  6   7  -5
 -7  5  -9  -3
 -3  7  -6  -1

### Problem 2a: 

Find the L1 norm of Matrix A, defined as: 

$$
\|A\|_1
=
\max_{1 \leq j \leq n}
\sum_{i=1}^{m}
|a_{ij}|
$$

In [87]:
using LinearAlgebra

opnorm(A, 1)  # To find the L1 norm, sum down each column and take the max

22.0

### Problem 2b: 

Find the Infinity Norm for Matrix A, defined as: 

$$
\|A\|_\infty
=
\max_{1 \leq i \leq m}
\sum_{j=1}^{n}
|a_{ij}|
$$

In [69]:
using LinearAlgebra

opnorm(A, Inf)  # To find the Infinity norm, sum across each row and take the max

24.0

### Problem 2c:

Show that the infinity norm of a matrix is equal to the 1-norm of its transpose:

$$
\|A\|_\infty
=
\max_{1 \leq i \leq m}
\sum_{j=1}^{n}
|a_{ij}|
=
\|A^T\|_1.
$$

Since

$$
\|A^T\|_1
=
\max_{1 \leq i \leq m}
\sum_{j=1}^{n}
|a_{ij}|,
$$

it follows that

$$
\|A\|_\infty = \|A^T\|_1.
$$


### Problem 2d: 

Use the following Julia commands to find $\|A\|_2$.

```julia
using LinearAlgebra

A = [
    -5   6   7  -5
    -7   5  -9  -3
    -3   7  -6  -1
]

norm(A, 2)
```

In [70]:
norm(A,2)

19.849433241279208

### Probelm 2e: 

Let

$$
C = AA^T
$$

and use Julia to find the eigenvalues of $C$ as follows:

```julia
S = eigvals(A * A')
```

Compare the norm $\|A\|_2$ with the square root of the largest eigenvalue, $S[k]$, of $C$.

Make a conjecture based on this comparison, and test your conjecture numerically using three additional matrices.

In [81]:
function compareEig2Norm(A)
    display(A)
    C = A*A'
    S = eigvals(C)
    println("square root of largest eigenvalue of C is: ", sqrt(maximum(S)))
    println("Operator norm, using opnorm (A,2) is:      ",opnorm(A,2))
end

compareEig2Norm (generic function with 1 method)

In [82]:
compareEig2Norm(A)

3×4 Matrix{Int64}:
 -5  6   7  -5
 -7  5  -9  -3
 -3  7  -6  -1

square root of largest eigenvalue of C is: 15.911800439346157
Operator norm, using opnorm (A,2) is:      15.911800439346163


In [83]:
B = rand(1:20, 4, 4)
compareEig2Norm(B)


4×4 Matrix{Int64}:
 20  17  13   2
 19  20   1  20
  8  14  15  18
 17   8   1   6

square root of largest eigenvalue of C is: 52.49093232635704
Operator norm, using opnorm (A,2) is:      52.49093232635705


In [85]:
C = rand(1:20, 3, 4)  # Try a 3x4 matrix.  
compareEig2Norm(C)

3×4 Matrix{Int64}:
 12  15  18  16
  1   9   9   3
  4  12  12  20

square root of largest eigenvalue of C is: 41.617422401102225
Operator norm, using opnorm (A,2) is:      41.61742240110222


In [86]:
D = rand(1:20, 8, 10)  # Try a 8x10 matrix.  
compareEig2Norm(D)

8×10 Matrix{Int64}:
  8  13  18  10   2   6  17  13  14   5
 20   4  12   5   7  17   7  10   1   7
 10   3   7   2  19  17  10  17  16   2
 13  11   4   1   9   7   6   3  10   5
  9   1   3   5  16  14  18  12   6   1
  9  10   5   2  17  19  16   8   6  11
 13  18   9   9  13  18   2  10  10  13
  5  20   7   6  18  19  11   1   6  12

square root of largest eigenvalue of C is: 90.71762973183417
Operator norm, using opnorm (A,2) is:      90.71762973183415


### Answer to Question2e
Based on the tests above, it seems that the L2 norm is equal to the square root of the largest Eigenvalue of the matrix.  

### Problem 3

Consider the equation $Ax=b$, where

$$
A =
\begin{bmatrix}
4 & -1 & 0 \\
-1 & 4 & -1 \\
0 & -1 & 4
\end{bmatrix},
\qquad
b =
\begin{bmatrix}
4 \\
4 \\
10
\end{bmatrix}.
$$

### Problem 3a: 
Show that A is diagonally dominant. 

### Answer to Question 3a: 
Since the magnitude of each diagonal element is greater than the sum of the magnitides of the non-diagonal elements, matrix A is diagonally dominant.  This can be confirmed by inspection.  

### Problem 3b: 
Let A = L + D + U where L is strictly lower triangular, D is diagonal and U is strictly upper triangular. What are L, D and U for the matrix A?

### Answer to Question 3b: 

We decompose

$$
A =
\begin{bmatrix}
4 & -1 & 0 \\
-1 & 4 & -1 \\
0 & -1 & 4
\end{bmatrix}
$$

as

$$
A = L + D + U,
$$

where \(L\) is strictly lower triangular, \(D\) is diagonal, and \(U\) is strictly upper triangular.

Thus,

$$
L =
\begin{bmatrix}
0 & 0 & 0 \\
-1 & 0 & 0 \\
0 & -1 & 0
\end{bmatrix},
$$

$$
D =
\begin{bmatrix}
4 & 0 & 0 \\
0 & 4 & 0 \\
0 & 0 & 4
\end{bmatrix},
$$

and

$$
U =
\begin{bmatrix}
0 & -1 & 0 \\
0 & 0 & -1 \\
0 & 0 & 0
\end{bmatrix}.
$$

Therefore,

$$
A = L + D + U.
$$

### Problem 3c: 

The Jacobi iterative method is given by

$$
D x^{(k+1)}
=
-(L+U)x^{(k)} + b.
$$

Starting with

$$
x^{(0)}
=
\begin{bmatrix}
0 \\
0 \\
0
\end{bmatrix},
$$

perform two Jacobi iterations to compute

$$
x^{(1)}
\quad \text{and} \quad
x^{(2)}.
$$

In [104]:
using LinearAlgebra
using SparseArrays

A = [4 -1  0;
    -1  4 -1;
     0 -1  4]

b = [2; 4; 10]

x_j = [0; 0; 0]

L = tril(A, -1)          # strictly lower triangular
D = Matrix(Diagonal(diag(A)))
U = triu(A, 1)           # strictly upper triangular

for i=1:2

    x_new = D \ (b - (L + U) * x_j)
    
    x_j=x_new
    
    # println("x^($i) = ", x) - added for debug 

end

x_j

3-element Vector{Float64}:
 0.75
 1.75
 2.75

### Answer to Question 3c:  
See the output above for the value of x_j after two iterations. 

### Problem 3d: 

The Gauss–Seidel iterative method is given by

$$
(L+D)x^{(k+1)}
=
-Ux^{(k)} + b.
$$

Starting with

$$
x^{(0)}
=
\begin{bmatrix}
0 \\
0 \\
0
\end{bmatrix},
$$

perform two Gauss–Seidel iterations to compute

$$
x^{(1)}
\quad \text{and} \quad
x^{(2)}.
$$

In [107]:
using LinearAlgebra
using SparseArrays

A = [4 -1  0;
    -1  4 -1;
     0 -1  4]

b = [2; 4; 10]

x_gs = [0; 0; 0]

L = tril(A, -1)          # strictly lower triangular
D = Matrix(Diagonal(diag(A)))
U = triu(A, 1)           # strictly upper triangular

for i=1:2

    x_new = (L + D) \ (b - U * x_gs)
    
    x_gs=x_new
    
    # println("x^($i) = ", x) - added for debug 

end

x_gs

3-element Vector{Float64}:
 0.78125
 1.890625
 2.97265625

### Answer to Question 3d:  
See the output above for the value of x_gs after two iterations. 

### Question 3e: 
Compare how quickly each of the above iterative methods approach the exact solution $$x = (1, 2, 3)$$.

In [105]:
x_exact = [1;2;3]

println("Jacobi difference from exact after two iterations:")
display(x_j-x_exact)

Jacobi difference from exact after two iterations:


3-element Vector{Float64}:
 -0.25
 -0.25
 -0.25

In [106]:
println("Gauss_Seidel difference after two iterations:")
display(x_gs-x_exact)

Gauss_Seidel difference after two iterations:


3-element Vector{Float64}:
 -0.21875
 -0.109375
 -0.02734375

### Answer to Question 3e:  
See the output above for the convergence error after two iterations.  Gauss-Seidel gets closer.  